# 001 EDA Notebook

This notebook loads the FD001 training dataset, creates the Remaining Useful Life (RUL) target, and compares a few baseline regression models.

In [1]:
from pathlib import Path

import pandas as pd
from sklearn.ensemble import RandomForestRegressor
from sklearn.linear_model import LinearRegression
from sklearn.metrics import r2_score
from sklearn.model_selection import GridSearchCV, train_test_split

RANDOM_STATE = 42

## Load And Prepare The Dataset

The raw NASA CMAPSS text files contain extra whitespace, so the notebook trims the imported dataframe to the 26 expected columns.

In [2]:
candidate_paths = [
    Path("Dataset") / "train_FD001.txt",
    Path("..") / "Dataset" / "train_FD001.txt",
]

for path in candidate_paths:
    if path.exists():
        data_path = path
        break
else:
    raise FileNotFoundError("Could not locate Dataset/train_FD001.txt")

column_names = ["engine_id", "cycle", "op1", "op2", "op3"] + [
    f"s{i}" for i in range(1, 22)
]

df = pd.read_csv(
    data_path,
    sep=r"\s+",
    header=None,
).iloc[:, :26]

df.columns = column_names
df.head()

,engine_id,cycle,op1,op2,op3,s1,s2,s3,s4,s5,...,s12,s13,s14,s15,s16,s17,s18,s19,s20,s21
0,1,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,...,521.66,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190
1,1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,...,522.28,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236
2,1,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,...,522.42,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442
3,1,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,...,522.86,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739
4,1,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,...,522.19,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044


## Quick Profile

A compact dataset summary helps confirm the import before moving into feature engineering.

In [3]:
profile = pd.DataFrame(
    {
        "rows": [df.shape[0]],
        "columns": [df.shape[1]],
        "engines": [df["engine_id"].nunique()],
        "max_cycle": [df["cycle"].max()],
        "missing_values": [int(df.isna().sum().sum())],
    }
)

profile

,rows,columns,engines,max_cycle,missing_values
0,20631,26,100,362,0


## Create The RUL Target

RUL is calculated as the maximum cycle for each engine minus the current cycle.

In [4]:
max_cycle_per_engine = df.groupby("engine_id")["cycle"].transform("max")
df["RUL"] = max_cycle_per_engine - df["cycle"]

model_df = df.drop(columns=["engine_id"])
model_df.head()

,cycle,op1,op2,op3,s1,s2,s3,s4,s5,s6,...,s13,s14,s15,s16,s17,s18,s19,s20,s21,RUL
0,1,-0.0007,-0.0004,100.0,518.67,641.82,1589.70,1400.60,14.62,21.61,...,2388.02,8138.62,8.4195,0.03,392,2388,100.0,39.06,23.4190,191
1,2,0.0019,-0.0003,100.0,518.67,642.15,1591.82,1403.14,14.62,21.61,...,2388.07,8131.49,8.4318,0.03,392,2388,100.0,39.00,23.4236,190
2,3,-0.0043,0.0003,100.0,518.67,642.35,1587.99,1404.20,14.62,21.61,...,2388.03,8133.23,8.4178,0.03,390,2388,100.0,38.95,23.3442,189
3,4,0.0007,0.0000,100.0,518.67,642.35,1582.79,1401.87,14.62,21.61,...,2388.08,8133.83,8.3682,0.03,392,2388,100.0,38.88,23.3739,188
4,5,-0.0019,-0.0002,100.0,518.67,642.37,1582.85,1406.22,14.62,21.61,...,2388.04,8133.80,8.4294,0.03,393,2388,100.0,38.90,23.4044,187


## Train/Test Split

The feature matrix excludes the target column, and the split is made reproducible with a fixed random seed.

In [5]:
X = model_df.drop(columns=["RUL"])
y = model_df["RUL"]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
)

X_train.shape, X_test.shape

((16504, 25), (4127, 25))

## Baseline Models

Two simple regressors are trained first: linear regression and a default random forest.

In [6]:
linear_model = LinearRegression()
random_forest_model = RandomForestRegressor(random_state=RANDOM_STATE)

linear_model.fit(X_train, y_train)
random_forest_model.fit(X_train, y_train)

pred_linear = linear_model.predict(X_test)
pred_rf = random_forest_model.predict(X_test)

baseline_results = pd.DataFrame(
    {
        "model": ["Linear Regression", "Random Forest"],
        "r2_score": [
            r2_score(y_test, pred_linear),
            r2_score(y_test, pred_rf),
        ],
    }
).sort_values("r2_score", ascending=False)

baseline_results

,model,r2_score
1,Random Forest,0.716755
0,Linear Regression,0.654979


## Tune The Random Forest

A small grid search is used to improve the random forest configuration.

In [7]:
param_grid = {
    "n_estimators": [100, 200],
    "max_depth": [10, 20, None],
    "min_samples_split": [2, 5],
    "min_samples_leaf": [1, 2],
}

grid = GridSearchCV(
    estimator=RandomForestRegressor(random_state=RANDOM_STATE),
    param_grid=param_grid,
    cv=3,
    scoring="neg_mean_squared_error",
    n_jobs=-1,
)

grid.fit(X_train, y_train)

pred_tuned_rf = grid.best_estimator_.predict(X_test)

tuned_results = pd.DataFrame(
    {
        "model": ["Tuned Random Forest"],
        "r2_score": [r2_score(y_test, pred_tuned_rf)],
        "best_params": [str(grid.best_params_)],
        "cv_neg_mse": [grid.best_score_],
    }
)

tuned_results

,model,r2_score,best_params,cv_neg_mse
0,Tuned Random Forest,0.722952,"{'max_depth': 10, 'min_samples_leaf': 2, 'min_...",-1309.189877
